# Vector data acquisition 
Code for downloading building vector data  
-> copied from Gitbook SDS320 L02 - Data Acquisition > Vector and contextual data   

A) Accessing Overture Maps  
B) Accessing OSM 

In [1]:
# loading all libraries
import os
from pathlib import Path
import geoai
import geopandas as gpd
import leafmap.osm as osm

In [ ]:
# A) Overture  
# download the file

## Bounding box in EPSG:4326 (west, south, east, north), transformed from EPSG:2056
bbox = [8.30703444, 47.03765288, 8.32033877, 47.04674762]

## define output path in data folder
project_dir = Path.cwd()
if project_dir.name in ["notebooks", "code"]:
    project_dir = project_dir.parent

download_dir = project_dir / "data" / "raw" / "vector" / "overture"
download_dir.mkdir(parents=True, exist_ok=True)
output_path = download_dir / "overture_buildings.geojson"

geoai.download_overture_buildings(
    bbox=bbox,
    output=str(output_path),
    overture_type="building",
)

overture_buildings = gpd.read_file(output_path).to_crs(epsg=2056)
overture_buildings.to_file(output_path, driver="GeoJSON", index=False)

full_path = os.path.abspath(output_path)
file_size_mb = os.path.getsize(full_path) / (1024 * 1024)

print(f"Buildings saved to {full_path} (CRS: {overture_buildings.crs})")
print(f"File size: {file_size_mb:.2f} MB")

Buildings saved to c:\Users\Maja\Documents\04-Studium\SDS320\project\data\raw\vector\overture\overture_buildings.geojson (CRS: EPSG:2056)
File size: 1.01 MB


In [2]:
# Access OSM-style data

# Query by bounding box
bbox = [8.30703444, 47.03765288, 8.32033877, 47.04674762]

buildings_osm = osm.quackosm_gdf_from_bbox(bbox, tags={"building": True})
buildings_osm = buildings_osm.to_crs(epsg=2056)

print(f"Downloaded {len(buildings_osm)} buildings; CRS: {buildings_osm.crs}")
project_dir = Path.cwd()
if project_dir.name in ["notebooks", "code"]:
    project_dir = project_dir.parent
download_dir = project_dir / "data" / "raw" / "vector" / "osm"
download_dir.mkdir(parents=True, exist_ok=True)
output_path = download_dir / "OSM_buildings.geojson"
buildings_osm.to_file(output_path, driver="GeoJSON", index=False)
print(f"OSM buildings saved to {output_path.resolve()}")
buildings_osm.head()

# Query by place name
#roads = osm.quackosm_gdf_from_place("Basel, Switzerland", tags={"highway": True})

#print(f"Downloaded {len(roads)} road segments")
#roads.head()

# more precise control over query area with Shapely geometry
#from shapely.geometry import Polygon

#polygon = Polygon([(8.53, 47.37),
#        (8.54, 47.37),
#        (8.54, 47.38),
#        (8.53, 47.38),])

#natural = osm.quackosm_gdf_from_geometry(polygon, tags={"natural": True})

#print(f"Downloaded {len(natural)} natural features")
#natural.head()

Output()

Finished operation in 0:01:37

Downloaded 765 buildings; CRS: EPSG:2056
OSM buildings saved to C:\Users\Maja\Documents\04-Studium\SDS320\project\data\raw\vector\osm\OSM_buildings.geojson


,building,geometry
feature_id,,
way/531260813,residential,"POLYGON ((2666198.46 1209935.72, 2666187.977 1..."
way/89286065,apartments,"POLYGON ((2666617 1209968.564, 2666623.853 120..."
way/89295101,apartments,"POLYGON ((2666045.503 1209980.109, 2666036.703..."
way/89307141,yes,"POLYGON ((2666818.372 1209990.739, 2666805.363..."
way/89295102,apartments,"POLYGON ((2666008.969 1209983.273, 2666000.285..."


## Visualization
Run the preview cell first, then the overlay cell to compare each raster tile with both building datasets.

In [1]:
from pathlib import Path
import geoai
import rasterio
from rasterio.enums import Resampling

project_dir = Path.cwd()
if project_dir.name in ["notebooks", "code"]:
    project_dir = project_dir.parent

vexcel_dir = project_dir / "data" / "raw" / "raster" / "vexcel"
tile_dirs = sorted(vexcel_dir.glob("colab_rgb_ortho*/tiles"))
if len(tile_dirs) != 1:
    raise FileNotFoundError(f"Expected one orthophoto tiles folder, found {len(tile_dirs)}: {tile_dirs}")

tile_dir = tile_dirs[0]
tile_paths = sorted(tile_dir.glob("*.tif"))
if not tile_paths:
    raise FileNotFoundError(f"No TIFF files found in {tile_dir}")

for tile_path in tile_paths:
    with rasterio.open(tile_path) as src:
        if src.count < 3:
            raise ValueError(f"Expected at least 3 RGB bands in {tile_path}, found {src.count}")
        scale = min(1.0, 900 / max(src.height, src.width))
        preview = src.read(
            (1, 2, 3),
            out_shape=(3, max(1, round(src.height * scale)), max(1, round(src.width * scale))),
            resampling=Resampling.average,
        ).transpose(1, 2, 0)
    geoai.view_image(preview, title=tile_path.name)

: 

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import geopandas as gpd

overture_path = project_dir / "data" / "raw" / "vector" / "overture" / "overture_buildings.geojson"
osm_path = project_dir / "data" / "raw" / "vector" / "OSM_buildings.geojson"
for vector_path in (overture_path, osm_path):
    if not vector_path.is_file():
        raise FileNotFoundError(f"Vector dataset not found: {vector_path}")

with rasterio.open(tile_paths[0]) as src:
    raster_crs = src.crs
if raster_crs is None:
    raise ValueError(f"Raster has no CRS: {tile_paths[0]}")

overture_buildings = gpd.read_file(overture_path).to_crs(raster_crs)
osm_buildings = gpd.read_file(osm_path).to_crs(raster_crs)

ncols = 5
nrows = math.ceil(len(tile_paths) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
for ax, tile_path in zip(axes.flat, tile_paths):
    with rasterio.open(tile_path) as src:
        if src.crs != raster_crs:
            raise ValueError(f"Tile CRS {src.crs} does not match {raster_crs}: {tile_path}")
        scale = min(1.0, 700 / max(src.height, src.width))
        rgb = src.read(
            (1, 2, 3),
            out_shape=(3, max(1, round(src.height * scale)), max(1, round(src.width * scale))),
            resampling=Resampling.average,
        ).transpose(1, 2, 0)
        left, bottom, right, top = src.bounds

    ax.imshow(rgb, extent=(left, right, bottom, top), origin="upper")
    for buildings, color in ((overture_buildings, "magenta"), (osm_buildings, "cyan")):
        visible = buildings.cx[left:right, bottom:top]
        if not visible.empty:
            visible.boundary.plot(ax=ax, color=color, linewidth=0.6)
    ax.set_title(tile_path.name)
    ax.set_axis_off()

for ax in axes.ravel()[len(tile_paths):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        Line2D([0], [0], color="magenta", lw=1.5, label="Overture buildings"),
        Line2D([0], [0], color="cyan", lw=1.5, label="OSM buildings"),
    ],
    loc="lower center",
    ncol=2,
)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()